# Smartphone Price Prediction: Exploratory Data Analysis & Modeling
This notebook demonstrates the end-to-end exploratory analysis, data cleaning, feature engineering, and model validation on the Coa-12.csv dataset.


In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, os.path.abspath('..'))
from src.data_loader import load_raw_data
from src.data_cleaning import perform_data_cleaning
from src.feature_engineering import engineer_features
from src.feature_selection import select_features
from src.data_splitting import split_data
from src.predict import load_model_bundle, predict_smartphone_price


## 1. Load and Inspect Raw Data

In [ ]:
raw_df = load_raw_data('../data/raw/Coa-12.csv')
print('Raw Shape:', raw_df.shape)
raw_df.head(3)


## 2. Data Cleaning and Quality Validation
We verify structural zeros in processor cluster 3, check hardware bounds, and report outliers without arbitrary deletion.


In [ ]:
clean_df, quality_report, summary = perform_data_cleaning(
    raw_df,
    output_cleaned_path='../data/processed/cleaned_smartphones.csv',
    output_quality_report_path='../reports/data_quality_report.csv',
    output_summary_path='../reports/cleaning_summary.txt'
)
print('Cleaned records preserved:', clean_df.shape[0])
quality_report.head(10)


## 3. Exploratory Data Analysis
Visualizing price distribution, correlation with semiconductor node, and RAM capacity.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(clean_df['Price_INR'], kde=True, ax=axes[0], color='#0284c7')
axes[0].set_title('Smartphone Price Distribution (₹ INR)')
axes[0].set_xlabel('Price (₹ INR)')

sns.boxplot(data=clean_df, x='RAM_Size_GB', y='Price_INR', ax=axes[1], palette='Blues_r', hue='RAM_Size_GB', legend=False)
axes[1].set_title('RAM Size (GB) vs. Price (₹ INR)')
plt.tight_layout()
plt.show()


## 4. Feature Engineering & Selection
Deriving max core frequency, weighted core frequency, total pixels, display category, and rear camera MP.


In [ ]:
eng_df = engineer_features(clean_df)
X, y, num_cols, cat_cols = select_features(eng_df, report_path='../reports/feature_selection_report.md')
print(f'Selected {len(num_cols)} numerical features and {len(cat_cols)} categorical features.')
X.head(3)


## 5. Model Evaluation and Final Test Predictions

In [ ]:
bundle = load_model_bundle('../models/best_smartphone_price_model.joblib')
print('Winning Model:', bundle['model_name'])
print('Test Metrics:', bundle['test_metrics'])


## 6. Live Inference Example

In [ ]:
sample_spec = {
    'Phone_Company': 'OnePlus',
    'RAM_Type': 'LPDDR5X',
    'RAM_Size_GB': 12,
    'Storage_GB': 256,
    'Battery_Capacity_mAh': 6000,
    'Process_Technology_nm': 3,
    'Display_Resolution_Width_px': 1264,
    'Display_Resolution_Height_px': 2780,
    'Refresh_Rate_Hz': 120,
    'Display_Type': 'AMOLED, 120Hz',
    'Processor_Core1_Count': 1,
    'Processor_Core1_Frequency_GHz': 3.3,
    'Processor_Core2_Count': 5,
    'Processor_Core2_Frequency_GHz': 3.1,
    'Processor_Core3_Count': 2,
    'Processor_Core3_Frequency_GHz': 2.3,
    'Camera_Specifications': 'Rear: 50 MP | Front: 32 MP'
}
res = predict_smartphone_price(sample_spec, model_bundle=bundle)
print('Predicted Price:', res['formatted_price'])
print('Valuation Range:', res['formatted_range'])
